# AISC DeepFake — Calibrated Weighted Fusion
## AUTO Artifact / Validation Adapter

Bu sürümde **validation_predictions.csv zorunlu değildir**.

Her model-bölge için validation skorları şu sırayla elde edilir:

1. Deney klasöründeki mevcut validation prediction CSV/NPZ artifact'ı
2. Mevcut feature/score NPZ artifact'ı
3. SwinV2 Tiny / EfficientNet-B0 için `best.ckpt + validation manifest` ile
   yalnızca validation inference
4. Hiçbiri mümkün değilse açık hata raporu

Ardından V4 yöntemi korunur:

`validation score -> Platt calibration -> validation ROC-AUC weight -> TEST calibrated score -> Eye+Brow+Mouth late fusion`

TEST seti calibration veya ağırlık öğrenmek için kullanılmaz.

In [1]:
# ============================================================
# 1) COLAB + CONFIG
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

!pip -q install scikit-learn pandas numpy pillow pyyaml tqdm torchvision

from pathlib import Path
from datetime import datetime, timezone

import json
import math
import os
import random
import re
import warnings

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

METHOD_NAME = "03_calibrated_weighted_fusion_auto_artifacts"
DECISION_THRESHOLD = 0.50
FRAME_AGGREGATION = "mean"
RELIABILITY_FLOOR = 1e-6
FIGURE_DPI = 600

ROOT = Path(
    "/content/drive/MyDrive/"
    "AISC DeepFake Çalışmaları/Deney 1"
)

RESULT_ROOTS = {
    "eye": ROOT / "Kader/Deney 1/Sonuçlar",
    "brow": ROOT / "Nazlıcan/Deney 1/Sonuçlar",
    "mouth": ROOT / "Dilara/Deney 1/Sonuçlar",
}

EXPERIMENT_ROOTS = {
    "swinv2_tiny": {
        "eye": RESULT_ROOTS["eye"] / "20260807_1031_eye_swinv2_tiny_seed42",
        "brow": RESULT_ROOTS["brow"] / "Kas_SwinV2_Tiny_Detayli_Sonuc_pdf",
        "mouth": RESULT_ROOTS["mouth"] / "20260807_2235_mouth_swinv2_tiny_seed42",
    },
    "efficientnet_b0": {
        "eye": RESULT_ROOTS["eye"] / "20260808_0803_eye_efficientnet_b0_seed42",
        "brow": RESULT_ROOTS["brow"] / "20260808_1248_eyebrow_efficientnet_b0_seed42",
        "mouth": RESULT_ROOTS["mouth"] / "20260808_1257_mouth_efficientnet_b0_seed42",
    },
    "swinv2_texture": {
        "eye": RESULT_ROOTS["eye"] / "20260806_1748_eye_swinv2_texturefusion_seed42",
        "brow": RESULT_ROOTS["brow"] / "Swin V2-Tiny + LBP + GLCM + Gabor + Wavelet Fusion",
        "mouth": RESULT_ROOTS["mouth"] / "SwinV2_TextureFusion_Mouth/20260807_1550_mouth_swinv2_texturefusion_seed42",
    },
}

TEST_PATHS = {
    "swinv2_tiny": {
        "eye": EXPERIMENT_ROOTS["swinv2_tiny"]["eye"] / "predictions/test_frame_predictions.csv",
        "brow": EXPERIMENT_ROOTS["swinv2_tiny"]["brow"] / "predictions/test_frame_predictions.csv",
        "mouth": EXPERIMENT_ROOTS["swinv2_tiny"]["mouth"] / "predictions/test_frame_predictions.csv",
    },
    "efficientnet_b0": {
        "eye": EXPERIMENT_ROOTS["efficientnet_b0"]["eye"] / "predictions/test_predictions.csv",
        "brow": EXPERIMENT_ROOTS["efficientnet_b0"]["brow"] / "predictions/test_predictions.csv",
        "mouth": EXPERIMENT_ROOTS["efficientnet_b0"]["mouth"] / "predictions/test_predictions.csv",
    },
    "swinv2_texture": {
        "eye": EXPERIMENT_ROOTS["swinv2_texture"]["eye"] / "full/predictions/test_predictions.csv",
        "brow": EXPERIMENT_ROOTS["swinv2_texture"]["brow"] / "predictions/test_predictions_frame_level.csv",
        "mouth": EXPERIMENT_ROOTS["swinv2_texture"]["mouth"] / "full/predictions/test_predictions.csv",
    },
}

OUTPUT_ROOT = RESULT_ROOTS["eye"] / "Fusion_Experiments"

RUN_ID = datetime.now(timezone.utc).strftime(
    "%Y%m%d_%H%M%S_auto_calibrated_weighted_seed42"
)

RUN_DIR = OUTPUT_ROOT / METHOD_NAME / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)

print("RUN:", RUN_DIR)

Mounted at /content/drive
RUN: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Kader/Deney 1/Sonuçlar/Fusion_Experiments/03_calibrated_weighted_fusion_auto_artifacts/20260809_195733_auto_calibrated_weighted_seed42


In [2]:
# ============================================================
# 2) IMPORTS + UTILITIES
# ============================================================

import torch
import torch.nn as nn

from PIL import Image
from tqdm.auto import tqdm

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, swin_v2_t

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("DEVICE:", DEVICE)


def atomic_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    check = pd.read_csv(tmp)
    if len(check) != len(df):
        raise RuntimeError(f"CSV verification failed: {path}")
    os.replace(tmp, path)


def atomic_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)
    with tmp.open("r", encoding="utf-8") as f:
        json.load(f)
    os.replace(tmp, path)

DEVICE: cuda


In [3]:
# ============================================================
# 3) COMMON SCHEMA / FRAME NORMALIZATION
# ============================================================

LABEL_CANDIDATES = [
    "target", "label_int", "label", "true_label",
    "y_true", "ground_truth"
]

PROB_CANDIDATES = [
    "probability_fake", "prob_fake", "fake_probability",
    "fake_prob", "probability", "prob", "score", "y_score"
]

KEY_CANDIDATES = [
    "source_frame", "relative_frame_path", "frame_path",
    "original_frame", "image_path", "path", "frame_stem"
]


def first_existing(columns, candidates):
    mapping = {str(c).lower(): c for c in columns}
    for candidate in candidates:
        if candidate.lower() in mapping:
            return mapping[candidate.lower()]
    return None


def normalize_label(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, (int, float, np.integer, np.floating)):
        return int(float(value) >= 0.5)

    value = str(value).strip().lower()

    if value in {"1", "fake", "deepfake", "manipulated", "sahte"}:
        return 1

    if value in {"0", "real", "genuine", "original", "gerçek"}:
        return 0

    return int(float(value) >= 0.5)


def canonical_frame_key(value):
    if pd.isna(value):
        return None

    text = str(value).replace("\\", "/").lower()
    filename = text.split("/")[-1]

    filename = re.sub(
        r"\.(jpg|jpeg|png|bmp|webp|npy)$",
        "",
        filename,
    )

    matches = re.findall(
        r"(real|fake)_(train|test|val|validation)_(\d+)",
        filename,
    )

    if not matches:
        return None

    label, split, number = matches[-1]

    if split == "validation":
        split = "val"

    return (
        f"{label}_{split}_"
        f"{str(int(number)).zfill(5)}"
    )

In [4]:
# ============================================================
# 4) VALIDATION ARTIFACT DISCOVERY
# ============================================================

def bounded_roots(exp_root):
    exp_root = Path(exp_root)

    roots = [
        exp_root,
        exp_root / "full",
        exp_root.parent if exp_root.name.lower() == "full" else exp_root,
    ]

    unique = []

    for root in roots:
        if root not in unique:
            unique.append(root)

    return unique


def discover_validation_artifacts(exp_root):
    """
    Sadece deney klasörü içinde arar.
    Drive kökünde rglob yapmaz.
    """

    records = []

    for root in bounded_roots(exp_root):

        if not root.is_dir():
            continue

        # Prediction CSV
        for directory in [
            root / "predictions",
            root / "artifacts",
            root / "features",
            root,
        ]:
            if not directory.is_dir():
                continue

            for p in directory.glob("*.csv"):
                name = p.name.lower()

                if "test" in name:
                    continue

                if (
                    "validation" in name
                    or re.search(
                        r"(^|[_\-.])val([_\-.]|$)",
                        name,
                    )
                ):
                    records.append(
                        {
                            "path": p,
                            "kind": "csv",
                        }
                    )

            for p in directory.glob("*.npz"):
                name = p.name.lower()

                if (
                    "val" in name
                    or "validation" in name
                    or "feature" in name
                ):
                    records.append(
                        {
                            "path": p,
                            "kind": "npz",
                        }
                    )

    # dedup
    out = []
    seen = set()

    for record in records:
        key = str(record["path"])

        if key not in seen:
            out.append(record)
            seen.add(key)

    return out


for family, region_roots in EXPERIMENT_ROOTS.items():
    print("\n", family)

    for region, root in region_roots.items():
        artifacts = discover_validation_artifacts(root)

        print(
            region,
            "->",
            [
                str(x["path"].name)
                for x in artifacts[:10]
            ],
        )


 swinv2_tiny
eye -> []
brow -> []
mouth -> []

 efficientnet_b0
eye -> []
brow -> []
mouth -> []

 swinv2_texture
eye -> []
brow -> ['texture_features_train.npz', 'texture_features_val.npz', 'texture_features_test.npz']
mouth -> []


In [5]:
# ============================================================
# 5) READ VALIDATION SCORE ARTIFACT IF POSSIBLE
# ============================================================

def dataframe_to_standard_val(df):
    label_col = first_existing(
        df.columns,
        LABEL_CANDIDATES,
    )

    prob_col = first_existing(
        df.columns,
        PROB_CANDIDATES,
    )

    if label_col is None or prob_col is None:
        return None

    key_col = first_existing(
        df.columns,
        KEY_CANDIDATES,
    )

    out = pd.DataFrame(
        {
            "label": df[label_col].map(
                normalize_label
            ).astype(int),
            "probability_fake": pd.to_numeric(
                df[prob_col],
                errors="coerce",
            ),
        }
    )

    if key_col is not None:
        out["fusion_key"] = df[key_col].map(
            canonical_frame_key
        )
    else:
        out["fusion_key"] = None

    if out["probability_fake"].isna().any():
        return None

    if (
        (out["probability_fake"] < 0)
        | (out["probability_fake"] > 1)
    ).any():
        return None

    return out


def try_read_csv_artifact(path):
    try:
        df = pd.read_csv(path)
    except Exception:
        return None

    return dataframe_to_standard_val(df)


def array_candidate(npz, names):
    lower = {
        key.lower(): key
        for key in npz.files
    }

    for name in names:
        if name.lower() in lower:
            return np.asarray(
                npz[lower[name.lower()]]
            )

    return None


def try_read_npz_artifact(path):
    try:
        z = np.load(
            path,
            allow_pickle=True,
        )
    except Exception:
        return None

    labels = array_candidate(
        z,
        [
            "labels", "y", "y_val", "val_labels",
            "targets", "target", "label"
        ],
    )

    probs = array_candidate(
        z,
        [
            "probability_fake", "prob_fake",
            "probabilities", "probs",
            "val_probabilities", "val_probs",
            "scores", "val_scores"
        ],
    )

    if labels is None or probs is None:
        return None

    labels = np.asarray(labels).reshape(-1)

    probs = np.asarray(probs)

    if probs.ndim == 2 and probs.shape[1] == 2:
        probs = probs[:, 1]

    probs = probs.reshape(-1)

    if len(labels) != len(probs):
        return None

    # decision scores are not automatically treated as probabilities.
    if (
        not np.isfinite(probs).all()
        or np.min(probs) < 0
        or np.max(probs) > 1
    ):
        return None

    return pd.DataFrame(
        {
            "fusion_key": None,
            "label": [
                normalize_label(x)
                for x in labels
            ],
            "probability_fake": probs.astype(float),
        }
    )


def load_existing_validation_scores(exp_root):
    artifacts = discover_validation_artifacts(
        exp_root
    )

    for record in artifacts:

        path = record["path"]

        if record["kind"] == "csv":
            result = try_read_csv_artifact(
                path
            )
        else:
            result = try_read_npz_artifact(
                path
            )

        if result is not None and len(result) > 0:
            return (
                result,
                {
                    "source": "existing_artifact",
                    "path": str(path),
                },
            )

    return None, None

In [6]:
# ============================================================
# 6) MANIFEST RESOLUTION FOR FALLBACK VALIDATION INFERENCE
# ============================================================

MANIFEST_NAMES = [
    "validated_training_manifest.csv",
    "training_manifest.csv",
    "metadata_used.csv",
    "eligible_metadata.csv",
    "eligible_mouth_metadata.csv",
]

IMAGE_COLUMNS = [
    "_resolved_image_path",
    "resolved_image_path",
    "combined_eye_path",
    "output_path",
    "training_mouth_path",
    "mouth_path",
    "image_path",
    "path",
]


def manifest_candidates(exp_root):
    paths = []

    for root in bounded_roots(exp_root):
        for name in MANIFEST_NAMES:
            paths += [
                root / "audit" / name,
                root / "artifacts" / name,
                root / name,
            ]

    return paths


def find_split_column(df):
    for c in ["split", "dataset_split", "subset", "set"]:
        if c in df.columns:
            return c
    return None


def find_image_column(df):
    for c in IMAGE_COLUMNS:
        if c not in df.columns:
            continue

        values = (
            df[c]
            .dropna()
            .astype(str)
            .head(100)
        )

        if values.empty:
            continue

        fraction = values.map(
            lambda x: Path(x).is_file()
        ).mean()

        if fraction >= 0.80:
            return c

    return None


def find_manifest(exp_root):
    diagnostics = []

    for p in manifest_candidates(exp_root):

        if not p.is_file():
            continue

        try:
            df = pd.read_csv(p)
        except Exception:
            continue

        split_col = find_split_column(df)
        label_col = first_existing(
            df.columns,
            LABEL_CANDIDATES,
        )
        image_col = find_image_column(df)

        diagnostics.append(
            {
                "path": str(p),
                "split_col": split_col,
                "label_col": label_col,
                "image_col": image_col,
            }
        )

        if (
            split_col is not None
            and label_col is not None
            and image_col is not None
        ):
            work = df.copy()

            work["_split"] = (
                work[split_col]
                .astype(str)
                .str.lower()
                .replace(
                    {
                        "validation": "val",
                        "valid": "val",
                    }
                )
            )

            work["_label"] = (
                work[label_col]
                .map(normalize_label)
                .astype(int)
            )

            work["_image_path"] = (
                work[image_col]
                .astype(str)
            )

            return work, {
                "path": str(p),
                "split_col": split_col,
                "label_col": label_col,
                "image_col": image_col,
            }

    raise FileNotFoundError(
        "Usable manifest not found.\n"
        + json.dumps(
            diagnostics,
            indent=2,
            ensure_ascii=False,
        )
    )

In [7]:
# ============================================================
# 7) SIMPLE BACKBONE VALIDATION INFERENCE
#    SWINV2 TINY + EFFICIENTNET-B0
# ============================================================

IMAGENET_MEAN = (
    0.485,
    0.456,
    0.406,
)

IMAGENET_STD = (
    0.229,
    0.224,
    0.225,
)


class ValImageDataset(Dataset):

    def __init__(
        self,
        df,
        image_size=224,
    ):
        self.df = (
            df.reset_index(
                drop=True
            )
        )

        self.transform = transforms.Compose(
            [
                transforms.Resize(
                    (
                        image_size,
                        image_size,
                    ),
                    interpolation=(
                        transforms.InterpolationMode.BICUBIC
                    ),
                    antialias=True,
                ),
                transforms.ToTensor(),
                transforms.Normalize(
                    IMAGENET_MEAN,
                    IMAGENET_STD,
                ),
            ]
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        path = Path(
            row["_image_path"]
        )

        image = Image.open(
            path
        ).convert(
            "RGB"
        )

        return (
            self.transform(image),
            int(row["_label"]),
            str(path),
        )


def find_best_checkpoint(exp_root):
    candidates = []

    for root in bounded_roots(exp_root):
        candidates += [
            root / "checkpoints/finetune/best.ckpt",
            root / "checkpoints/best.ckpt",
        ]

    for p in candidates:
        if p.is_file():
            return p

    return None


def checkpoint_state(path):
    obj = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    if isinstance(obj, dict):
        for key in [
            "model_state_dict",
            "state_dict",
            "model",
        ]:
            if (
                key in obj
                and isinstance(
                    obj[key],
                    dict,
                )
            ):
                return obj[key], obj

    if isinstance(obj, dict) and all(
        torch.is_tensor(v)
        for v in obj.values()
    ):
        return obj, {}

    raise RuntimeError(
        f"state_dict not found: {path}"
    )


def remove_uniform_prefix(state, prefix):
    if state and all(
        key.startswith(prefix)
        for key in state
    ):
        return {
            key[len(prefix):]: value
            for key, value in state.items()
        }

    return state


class SwinWrapper(nn.Module):

    def __init__(self):
        super().__init__()

        self.backbone = swin_v2_t(
            weights=None
        )

        dim = (
            self.backbone
            .head
            .in_features
        )

        self.backbone.head = (
            nn.Identity()
        )

        self.classifier = (
            nn.Sequential(
                nn.LayerNorm(dim),
                nn.Dropout(0.2),
                nn.Linear(
                    dim,
                    1,
                ),
            )
        )

    def forward(self, x):
        return (
            self.classifier(
                self.backbone(x)
            )
            .squeeze(1)
        )


def build_efficientnet_from_checkpoint(state):
    """
    Supports the observed checkpoint format where keys begin with:
    features.* / classifier.*
    instead of backbone.features.*.
    """

    model = efficientnet_b0(
        weights=None
    )

    # Infer final output dimension from checkpoint.
    classifier_weight_keys = [
        k
        for k in state
        if (
            k.startswith(
                "classifier."
            )
            and k.endswith(
                ".weight"
            )
            and state[k].ndim == 2
        )
    ]

    if not classifier_weight_keys:
        raise RuntimeError(
            "EfficientNet classifier weight not found in checkpoint."
        )

    classifier_weight_keys = sorted(
        classifier_weight_keys
    )

    final_key = (
        classifier_weight_keys[-1]
    )

    out_features = int(
        state[final_key].shape[0]
    )

    in_features = int(
        state[final_key].shape[1]
    )

    # Common binary setup.
    if out_features == 1:
        model.classifier = (
            nn.Sequential(
                nn.Dropout(
                    p=0.2,
                    inplace=True,
                ),
                nn.Linear(
                    in_features,
                    1,
                ),
            )
        )

    elif out_features == 2:
        model.classifier = (
            nn.Sequential(
                nn.Dropout(
                    p=0.2,
                    inplace=True,
                ),
                nn.Linear(
                    in_features,
                    2,
                ),
            )
        )

    else:
        raise RuntimeError(
            f"Unexpected EfficientNet output dimension: {out_features}"
        )

    return model, out_features


def load_model_for_family(
    family,
    checkpoint_path,
):
    state, meta = checkpoint_state(
        checkpoint_path
    )

    state = remove_uniform_prefix(
        state,
        "model.",
    )

    if family == "swinv2_tiny":

        # First try wrapper used by current Swin run.
        model = SwinWrapper()

        try:
            model.load_state_dict(
                state,
                strict=True,
            )

            return (
                model.to(DEVICE),
                1,
                "swin_wrapper",
            )

        except RuntimeError:
            pass

        # Fallback: direct torchvision Swin keys.
        direct = swin_v2_t(
            weights=None
        )

        head_keys = [
            k
            for k in state
            if (
                k.startswith("head.")
                and k.endswith(".weight")
                and state[k].ndim == 2
            )
        ]

        if head_keys:
            key = head_keys[-1]

            direct.head = nn.Linear(
                int(
                    state[key].shape[1]
                ),
                int(
                    state[key].shape[0]
                ),
            )

            direct.load_state_dict(
                state,
                strict=True,
            )

            return (
                direct.to(DEVICE),
                int(
                    state[key].shape[0]
                ),
                "swin_direct",
            )

        raise RuntimeError(
            "SwinV2 checkpoint architecture could not be reconstructed."
        )

    if family == "efficientnet_b0":

        model, out_dim = (
            build_efficientnet_from_checkpoint(
                state
            )
        )

        model.load_state_dict(
            state,
            strict=True,
        )

        return (
            model.to(DEVICE),
            out_dim,
            "efficientnet_direct",
        )

    raise ValueError(
        f"Validation inference fallback not implemented for {family}"
    )


def validation_inference(
    family,
    exp_root,
):
    manifest, manifest_audit = (
        find_manifest(exp_root)
    )

    val = manifest[
        manifest["_split"] == "val"
    ].copy()

    if val.empty:
        raise RuntimeError(
            "Validation split is empty."
        )

    missing = [
        path
        for path in val["_image_path"]
        if not Path(path).is_file()
    ]

    if missing:
        raise FileNotFoundError(
            f"{len(missing)} validation images are missing. "
            f"Example: {missing[:3]}"
        )

    ckpt = find_best_checkpoint(
        exp_root
    )

    if ckpt is None:
        raise FileNotFoundError(
            f"best.ckpt not found near {exp_root}"
        )

    model, output_dim, architecture = (
        load_model_for_family(
            family,
            ckpt,
        )
    )

    model.eval()

    loader = DataLoader(
        ValImageDataset(val),
        batch_size=32,
        shuffle=False,
        num_workers=0,
        pin_memory=(
            DEVICE.type == "cuda"
        ),
    )

    labels = []
    probs = []
    paths = []

    with torch.no_grad():

        for images, y, image_paths in tqdm(
            loader,
            desc=f"{family} validation",
        ):
            images = images.to(
                DEVICE
            )

            logits = model(
                images
            )

            if output_dim == 1:
                p = torch.sigmoid(
                    logits.reshape(-1)
                )

            else:
                p = torch.softmax(
                    logits,
                    dim=1,
                )[:, 1]

            probs.extend(
                p.detach()
                .cpu()
                .numpy()
                .tolist()
            )

            labels.extend(
                y.numpy()
                .astype(int)
                .tolist()
            )

            paths.extend(
                list(image_paths)
            )

    output = pd.DataFrame(
        {
            "image_path": paths,
            "fusion_key": [
                canonical_frame_key(x)
                for x in paths
            ],
            "label": labels,
            "probability_fake": probs,
        }
    )

    return output, {
        "source": "checkpoint_validation_inference",
        "checkpoint": str(ckpt),
        "architecture": architecture,
        "manifest": manifest_audit,
        "rows": len(output),
    }

In [8]:
# ============================================================
# 8) AUTO VALIDATION SCORE RESOLVER
# ============================================================

def resolve_validation_scores(
    family,
    region,
):
    exp_root = (
        EXPERIMENT_ROOTS[
            family
        ][region]
    )

    existing, audit = (
        load_existing_validation_scores(
            exp_root
        )
    )

    if existing is not None:

        print(
            family,
            region,
            "-> EXISTING ARTIFACT",
            audit["path"],
        )

        return existing, audit

    if family in {
        "swinv2_tiny",
        "efficientnet_b0",
    }:

        print(
            family,
            region,
            "-> CHECKPOINT + VAL MANIFEST"
        )

        return validation_inference(
            family,
            exp_root,
        )

    raise RuntimeError(
        f"{family}/{region}: no readable validation score artifact "
        "was found and generic texture fallback is intentionally "
        "not fabricated."
    )


VALIDATION_DATA = {}
VALIDATION_AUDIT = {}

resolver_rows = []

for family in EXPERIMENT_ROOTS:

    VALIDATION_DATA[family] = {}
    VALIDATION_AUDIT[family] = {}

    for region in (
        "eye",
        "brow",
        "mouth",
    ):

        try:

            data, audit = (
                resolve_validation_scores(
                    family,
                    region,
                )
            )

            VALIDATION_DATA[
                family
            ][region] = data

            VALIDATION_AUDIT[
                family
            ][region] = audit

            resolver_rows.append(
                {
                    "model_family": family,
                    "region": region,
                    "status": "SUCCESS",
                    "source": audit[
                        "source"
                    ],
                    "rows": len(
                        data
                    ),
                    "detail": str(
                        audit.get(
                            "path",
                            audit.get(
                                "checkpoint",
                                "",
                            ),
                        )
                    ),
                }
            )

        except Exception as exc:

            VALIDATION_DATA[
                family
            ][region] = None

            VALIDATION_AUDIT[
                family
            ][region] = {
                "source": "FAILED",
                "error": str(exc),
            }

            resolver_rows.append(
                {
                    "model_family": family,
                    "region": region,
                    "status": "FAILED",
                    "source": "",
                    "rows": 0,
                    "detail": (
                        f"{type(exc).__name__}: "
                        f"{exc}"
                    ),
                }
            )


RESOLVER_DF = pd.DataFrame(
    resolver_rows
)

atomic_csv(
    RESOLVER_DF,
    RUN_DIR
    / "audit"
    / "validation_score_resolution.csv",
)

display(
    RESOLVER_DF
)

swinv2_tiny eye -> CHECKPOINT + VAL MANIFEST
swinv2_tiny brow -> CHECKPOINT + VAL MANIFEST
swinv2_tiny mouth -> CHECKPOINT + VAL MANIFEST
efficientnet_b0 eye -> CHECKPOINT + VAL MANIFEST
efficientnet_b0 brow -> CHECKPOINT + VAL MANIFEST
efficientnet_b0 mouth -> CHECKPOINT + VAL MANIFEST


,model_family,region,status,source,rows,detail
0,swinv2_tiny,eye,FAILED,,0,FileNotFoundError: Usable manifest not found.\...
1,swinv2_tiny,brow,FAILED,,0,FileNotFoundError: Usable manifest not found.\...
2,swinv2_tiny,mouth,FAILED,,0,FileNotFoundError: Usable manifest not found.\...
3,efficientnet_b0,eye,FAILED,,0,FileNotFoundError: Usable manifest not found.\...
4,efficientnet_b0,brow,FAILED,,0,FileNotFoundError: Usable manifest not found.\...
5,efficientnet_b0,mouth,FAILED,,0,FileNotFoundError: Usable manifest not found.\...
6,swinv2_texture,eye,FAILED,,0,RuntimeError: swinv2_texture/eye: no readable ...
7,swinv2_texture,brow,FAILED,,0,RuntimeError: swinv2_texture/brow: no readable...
8,swinv2_texture,mouth,FAILED,,0,RuntimeError: swinv2_texture/mouth: no readabl...


In [ ]:
# ============================================================
# 9) TEST PREDICTION LOADER + REGION ALIGNMENT
# ============================================================

def load_test_region(
    path,
    region,
):
    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(
            path
        )

    df = pd.read_csv(
        path
    )

    label_col = first_existing(
        df.columns,
        LABEL_CANDIDATES,
    )

    prob_col = first_existing(
        df.columns,
        PROB_CANDIDATES,
    )

    if label_col is None or prob_col is None:
        raise ValueError(
            f"{region}: incompatible TEST schema: {list(df.columns)}"
        )

    # Direct key first.
    key = None

    for c in KEY_CANDIDATES:
        if c in df.columns:
            candidate = df[c].map(
                canonical_frame_key
            )

            if candidate.notna().mean() >= 0.95:
                key = candidate
                break

    # sample_id fallback via nearby metadata.
    if key is None and "sample_id" in df.columns:

        metadata = None

        for p in manifest_candidates(
            path.parent.parent
        ):
            if p.is_file():
                try:
                    m = pd.read_csv(p)
                except Exception:
                    continue

                if "sample_id" in m.columns:
                    metadata = m
                    break

        if metadata is not None:

            source_col = None

            for c in KEY_CANDIDATES:
                if c in metadata.columns:
                    k = metadata[c].map(
                        canonical_frame_key
                    )

                    if k.notna().mean() >= 0.95:
                        source_col = c
                        metadata = metadata.copy()
                        metadata["_fusion_key"] = k
                        break

            if source_col is not None:

                mapping = (
                    metadata[
                        [
                            "sample_id",
                            "_fusion_key",
                        ]
                    ]
                    .dropna()
                    .drop_duplicates(
                        "sample_id"
                    )
                )

                temp = pd.DataFrame(
                    {
                        "sample_id": (
                            df["sample_id"]
                            .astype(str)
                        )
                    }
                )

                mapping[
                    "sample_id"
                ] = (
                    mapping[
                        "sample_id"
                    ]
                    .astype(str)
                )

                key = (
                    temp.merge(
                        mapping,
                        on="sample_id",
                        how="left",
                    )[
                        "_fusion_key"
                    ]
                )

    if key is None or key.isna().any():
        raise ValueError(
            f"{region}: TEST frame keys could not be resolved."
        )

    work = pd.DataFrame(
        {
            "fusion_key": key,
            f"label_{region}": (
                df[label_col]
                .map(normalize_label)
                .astype(int)
            ),
            f"p_{region}": pd.to_numeric(
                df[prob_col],
                errors="raise",
            ),
        }
    )

    grouped = (
        work.groupby(
            "fusion_key",
            as_index=False,
        )
        .agg(
            **{
                f"label_{region}": (
                    f"label_{region}",
                    "first",
                ),
                f"p_{region}": (
                    f"p_{region}",
                    "mean",
                ),
            }
        )
    )

    return grouped


def align_test_family(
    family,
):
    e = load_test_region(
        TEST_PATHS[family]["eye"],
        "eye",
    )

    b = load_test_region(
        TEST_PATHS[family]["brow"],
        "brow",
    )

    m = load_test_region(
        TEST_PATHS[family]["mouth"],
        "mouth",
    )

    x = (
        e.merge(
            b,
            on="fusion_key",
            how="inner",
        )
        .merge(
            m,
            on="fusion_key",
            how="inner",
        )
    )

    if x.empty:
        raise RuntimeError(
            "Common TEST frame intersection is empty."
        )

    label_matrix = x[
        [
            "label_eye",
            "label_brow",
            "label_mouth",
        ]
    ]

    if not label_matrix.nunique(
        axis=1
    ).eq(1).all():
        raise RuntimeError(
            "TEST label disagreement across regions."
        )

    x["label"] = (
        label_matrix.iloc[:, 0]
        .astype(int)
    )

    return x

In [ ]:
# ============================================================
# 10) CALIBRATION + WEIGHTED FUSION
# ============================================================

def fit_platt(
    probability,
    label,
):
    model = LogisticRegression(
        C=1.0,
        solver="lbfgs",
        max_iter=2000,
        random_state=SEED,
    )

    model.fit(
        np.asarray(
            probability
        ).reshape(-1, 1),
        np.asarray(
            label
        ).astype(int),
    )

    return model


def calibrate(
    model,
    probability,
):
    return model.predict_proba(
        np.asarray(
            probability
        ).reshape(-1, 1)
    )[:, 1]


def compute_metrics(
    y,
    p,
):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)

    pred = (
        p >= DECISION_THRESHOLD
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y,
        pred,
        labels=[0, 1],
    ).ravel()

    return {
        "n": len(y),
        "accuracy": accuracy_score(
            y,
            pred,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y,
            pred,
        ),
        "precision": precision_score(
            y,
            pred,
            zero_division=0,
        ),
        "recall": recall_score(
            y,
            pred,
            zero_division=0,
        ),
        "specificity": (
            tn / (tn + fp)
            if (tn + fp)
            else np.nan
        ),
        "f1": f1_score(
            y,
            pred,
            zero_division=0,
        ),
        "roc_auc": roc_auc_score(
            y,
            p,
        ),
        "pr_auc": average_precision_score(
            y,
            p,
        ),
        "brier_score": brier_score_loss(
            y,
            p,
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


STATUS = []
METRIC_ROWS = []
WEIGHT_ROWS = []


for family in EXPERIMENT_ROOTS:

    print(
        "\n"
        + "=" * 100
    )

    print(
        "FAMILY:",
        family,
    )

    print(
        "=" * 100
    )

    missing = [
        region
        for region in (
            "eye",
            "brow",
            "mouth",
        )
        if VALIDATION_DATA[
            family
        ][region] is None
    ]

    if missing:

        STATUS.append(
            {
                "model_family": family,
                "status": (
                    "BLOCKED_VALIDATION_ARTIFACT"
                ),
                "missing_regions": (
                    ",".join(missing)
                ),
            }
        )

        print(
            "BLOCKED:",
            missing,
        )

        continue

    test = align_test_family(
        family
    )

    calibrators = {}
    aucs = {}
    reliability = {}

    for region in (
        "eye",
        "brow",
        "mouth",
    ):

        val = VALIDATION_DATA[
            family
        ][region]

        # Calibration is region-specific.
        calibrator = fit_platt(
            val[
                "probability_fake"
            ],
            val[
                "label"
            ],
        )

        calibrators[
            region
        ] = calibrator

        val_cal = calibrate(
            calibrator,
            val[
                "probability_fake"
            ],
        )

        auc = float(
            roc_auc_score(
                val[
                    "label"
                ],
                val_cal,
            )
        )

        aucs[
            region
        ] = auc

        reliability[
            region
        ] = max(
            auc - 0.5,
            RELIABILITY_FLOOR,
        )

        test[
            f"cal_p_{region}"
        ] = calibrate(
            calibrator,
            test[
                f"p_{region}"
            ],
        )

    denominator = sum(
        reliability.values()
    )

    weights = {
        region: (
            reliability[
                region
            ]
            / denominator
        )
        for region in (
            "eye",
            "brow",
            "mouth",
        )
    }

    test[
        "fusion_probability"
    ] = (
        weights["eye"]
        * test["cal_p_eye"]
        + weights["brow"]
        * test["cal_p_brow"]
        + weights["mouth"]
        * test["cal_p_mouth"]
    )

    family_dir = (
        RUN_DIR
        / family
    )

    atomic_csv(
        test,
        family_dir
        / "predictions"
        / "aligned_test_calibrated_fusion.csv",
    )

    family_metric_rows = []

    for name, column in {
        "eye_calibrated": "cal_p_eye",
        "brow_calibrated": "cal_p_brow",
        "mouth_calibrated": "cal_p_mouth",
        "fusion": "fusion_probability",
    }.items():

        row = {
            "model_family": family,
            "evaluation": name,
            "method": METHOD_NAME,
            "weight_eye": weights[
                "eye"
            ],
            "weight_brow": weights[
                "brow"
            ],
            "weight_mouth": weights[
                "mouth"
            ],
            **compute_metrics(
                test["label"],
                test[column],
            ),
        }

        family_metric_rows.append(
            row
        )

        METRIC_ROWS.append(
            row
        )

    atomic_csv(
        pd.DataFrame(
            family_metric_rows
        ),
        family_dir
        / "metrics"
        / "test_metrics.csv",
    )

    for region in (
        "eye",
        "brow",
        "mouth",
    ):
        WEIGHT_ROWS.append(
            {
                "model_family": family,
                "region": region,
                "validation_auc": (
                    aucs[region]
                ),
                "weight": (
                    weights[region]
                ),
                "validation_source": (
                    VALIDATION_AUDIT[
                        family
                    ][region][
                        "source"
                    ]
                ),
            }
        )

    fusion_row = [
        row
        for row in family_metric_rows
        if row[
            "evaluation"
        ] == "fusion"
    ][0]

    STATUS.append(
        {
            "model_family": family,
            "status": "SUCCESS",
            "common_test_frames": len(
                test
            ),
            "fusion_roc_auc": fusion_row[
                "roc_auc"
            ],
            "fusion_f1": fusion_row[
                "f1"
            ],
        }
    )

    print(
        "SUCCESS | "
        f"TEST={len(test)} | "
        f"ROC-AUC={fusion_row['roc_auc']:.4f} | "
        f"F1={fusion_row['f1']:.4f}"
    )


STATUS_DF = pd.DataFrame(
    STATUS
)

METRICS_DF = pd.DataFrame(
    METRIC_ROWS
)

WEIGHTS_DF = pd.DataFrame(
    WEIGHT_ROWS
)

atomic_csv(
    STATUS_DF,
    RUN_DIR
    / "family_status.csv",
)

if not METRICS_DF.empty:
    atomic_csv(
        METRICS_DF,
        RUN_DIR
        / "metrics"
        / "all_metrics.csv",
    )

if not WEIGHTS_DF.empty:
    atomic_csv(
        WEIGHTS_DF,
        RUN_DIR
        / "metrics"
        / "weights.csv",
    )

display(
    STATUS_DF
)

## Bu sürümde ne değişti?

- `validation_predictions.csv` zorunluluğu kaldırıldı.
- Önce mevcut CSV/NPZ validation artifact'ları okunuyor.
- SwinV2 Tiny ve EfficientNet-B0 için artifact yoksa `best.ckpt + val manifest`
  üzerinden validation inference otomatik yapılıyor.
- EfficientNet checkpoint'lerinde daha önce görülen `features.*` anahtar yapısı
  doğrudan torchvision EfficientNet olarak yükleniyor; `backbone.features.*`
  zorlanmıyor.
- Texture modelde mevcut validation score/NPZ artifact varsa doğrudan kullanılır.
  Mevcut artifact yoksa yanlış bir mimari tahminiyle sonuç uydurulmaz.